# Detecção de Fraude em Cartões de Crédito: Pipeline End-to-End
### Desbalanceamento Extremo, Modelagem Comparativa e Explicabilidade com SHAP

Neste notebook, construímos uma solução completa para detectar fraudes em transações financeiras usando o dataset clássico do Kaggle / TensorFlow (`creditcard.csv`).

#### Estrutura do Notebook:
1. **Configuração do Ambiente e Bibliotecas**: Instalação e importações necessárias.
2. **Exploração de Dados (EDA)**: Inspeção de dimensões, tipos, valores nulos e quantificação do desbalanceamento severo (0.17% de fraudes).
3. **Engenharia de Recursos e Pré-processamento**:
   - Transformação logarítmica e escalonamento robusto de `Amount` e `Time`.
   - Divisão estratificada em Treino e Teste (`stratify=y`).
4. **Estratégias para Desbalanceamento**:
   - Pesos de classe (`class_weight='balanced'`, `scale_pos_weight`).
   - Reamostragem: *Undersampling* aleatório e *SMOTE* (Oversampling sintético).
5. **Treinamento e Comparação de Modelos**:
   - Baseline: Regressão Logística.
   - Modelo Baseado em Árvores: Random Forest.
   - Modelo de Gradiente Boosting: XGBoost.
6. **Avaliação Focada em Métricas Corretas**:
   - Relatório de Classificação (Recall, Precision, F1-Score).
   - Curva ROC-AUC e Curva Precision-Recall (PR-AUC / Average Precision).
   - Otimização do Limiar de Decisão (*Threshold Tuning*).
   - Tabela comparativa consolidada.
7. **Explicabilidade do Modelo (XAI)**:
   - Importância global das variáveis.
   - Análise com SHAP (*SHapley Additive exPlanations*): impacto global (*beeswarm*) e diagnóstico local de uma transação fraudulenta individual (*waterfall plot*).
8. **Próximos Passos e Oportunidades de Evolução**.


In [ ]:
# Instalação de dependências essenciais (caso rode no Google Colab / novo ambiente)
!pip install -q imbalanced-learn xgboost shap seaborn matplotlib scikit-learn pandas numpy

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Divisão
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Balanceamento
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Métricas de Avaliação
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

# Explicabilidade
import shap

# Estilo visual dos gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('tab10')
pd.set_option('display.float_format', lambda x: '%.4f' % x)


## 1. Carregamento e Exploração de Dados (EDA)

Vamos carregar os dados diretamente da URL oficial do TensorFlow e analisar a distribuição das classes.


In [ ]:
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
print("Baixando e carregando o dataset...")
df = pd.read_csv(DATA_URL)

print(f"Dimensões do dataset: {df.shape[0]:,} linhas e {df.shape[1]} colunas.")
df.head()


In [ ]:
# Verificação de tipos de dados e valores nulos
print("Total de valores ausentes por coluna:", df.isnull().sum().max())

# Diagnóstico do Desbalanceamento
contagem_classes = df['Class'].value_counts()
pct_normal = (contagem_classes[0] / len(df)) * 100
pct_fraude = (contagem_classes[1] / len(df)) * 100

print(f"Transações Normais (0): {contagem_classes[0]:,} ({pct_normal:.3f}%)")
print(f"Transações Fraudulentas (1): {contagem_classes[1]:,} ({pct_fraude:.3f}%)")
print(f"Razão de desbalanceamento: ~1 fraude para cada {int(contagem_classes[0] / contagem_classes[1])} normais.")

# Visualização gráfica do desbalanceamento
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='Class', data=df, ax=ax[0], palette=['#2b5c8f', '#d9534f'])
ax[0].set_title("Distribuição Absoluta das Classes (Escala Log)", fontsize=13, fontweight='bold')
ax[0].set_yscale('log')
ax[0].set_xticklabels(['Normal (0)', 'Fraude (1)'])
ax[0].set_ylabel("Quantidade (Log Scale)")

# Comparação de valores de transações (Amount)
sns.boxplot(x='Class', y='Amount', data=df, ax=ax[1], palette=['#2b5c8f', '#d9534f'], showfliers=False)
ax[1].set_title("Distribuição de Valor por Classe (Sem Outliers)", fontsize=13, fontweight='bold')
ax[1].set_xticklabels(['Normal (0)', 'Fraude (1)'])
ax[1].set_ylabel("Valor da Transação ($)")

plt.tight_layout()
plt.show()


## 2. Preparação de Dados e Engenharia de Recursos

### Por que não podemos usar acurácia?
Com 99,83% de transações normais, um classificador ingênuo (*dummy*) que prevê sempre "não-fraude" terá **99,83% de acurácia**, mas taxa de detecção de fraudes igual a **0%**.
Em detecção de fraude, a prioridade máxima é o **Recall da classe 1** (quantas fraudes reais conseguimos capturar), equilibrado com uma **Precisão** aceitável para não sobrecarregar a equipe antifraude com falsos positivos.

### Passos de Preparação:
1. `Amount`: Distribuição altamente assimétrica com valores extremos. Aplicamos transformação logarítmica (`log1p`) e `RobustScaler` (mais resistente a outliers que StandardScaler).
2. `Time`: Transforma segundos em escala contínua padronizada.
3. `stratify=y`: Garante que a divisão de treino e teste preserve rigorosamente a mesma proporção de 0.17% de fraudes em ambos os conjuntos.


In [ ]:
# Cópia do dataframe para transformações
df_prep = df.copy()

# 1. Feature Engineering: Log do Amount e Escalonamento
rob_scaler = RobustScaler()
std_scaler = StandardScaler()

df_prep['log_amount'] = np.log1p(df_prep['Amount'])
df_prep['scaled_amount'] = rob_scaler.fit_transform(df_prep['Amount'].values.reshape(-1, 1))
df_prep['scaled_time'] = rob_scaler.fit_transform(df_prep['Time'].values.reshape(-1, 1))

# Removemos as colunas originais brutas para evitar redundância
df_prep.drop(['Time', 'Amount'], axis=1, inplace=True)

# 2. Separação de Features e Target
X = df_prep.drop('Class', axis=1)
y = df_prep['Class']

# 3. Divisão Estratificada (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Conjunto de Treino: {X_train.shape[0]:,} transações ({y_train.sum()} fraudes)")
print(f"Conjunto de Teste:  {X_test.shape[0]:,} transações ({y_test.sum()} fraudes)")
print(f"Taxa de fraude no Treino: {y_train.mean():.4%}")
print(f"Taxa de fraude no Teste:  {y_test.mean():.4%}")


## 3. Estratégias de Balanceamento e Reamostragem

Temos duas abordagens complementares:
1. **Algorítmica**: Atribuir maior penalidade para erros na classe minoritária via `class_weight='balanced'` ou `scale_pos_weight`.
2. **Baseada em Dados**:
   - **SMOTE (Synthetic Minority Over-sampling Technique)**: Cria transações fraudulentas sintéticas no espaço de features de treino.
   - **RandomUnderSampler**: Reduz a classe majoritária para agilizar e equilibrar o treinamento.

*Atenção Crítica:* Toda e qualquer técnica de reamostragem deve ser aplicada **exclusivamente nos dados de treino**, nunca no conjunto de teste!


In [ ]:
# Criando conjuntos alternativos para testes de reamostragem
# 1. SMOTE (Oversampling no Treino)
print("Gerando amostras sintéticas com SMOTE...")
smote = SMOTE(random_state=42, sampling_strategy=0.1) # Eleva fraudes para 10% da classe majoritária
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 2. Random Undersampling
print("Gerando conjunto balanceado via Undersampling...")
rus = RandomUnderSampler(random_state=42, sampling_strategy=0.5) # 1 fraude para cada 2 normais
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print(f"Distribuição Treino Original:       {y_train.value_counts().to_dict()}")
print(f"Distribuição Treino com SMOTE:      {y_train_smote.value_counts().to_dict()}")
print(f"Distribuição Treino com Undersample:{y_train_rus.value_counts().to_dict()}")


## 4. Treinamento e Comparação de Modelos

Treinaremos e avaliaremos três arquiteturas fundamentais:
1. **Baseline**: Regressão Logística com `class_weight='balanced'`.
2. **Random Forest**: Floresta aleatória com `class_weight='balanced'`.
3. **XGBoost**: Gradient Boosting com `scale_pos_weight` calculado proporcionalmente.


In [ ]:
# Dicionário para armazenar modelos e predições
modelos = {}
predicoes = {}
probabilidades = {}

# 1. Baseline: Regressão Logística com Class Weight Balanced
print("Treinando 1/3: Regressão Logística...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
modelos['Regressão Logística (Balanced)'] = lr

# 2. Random Forest com Class Weight Balanced
print("Treinando 2/3: Random Forest...")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)
modelos['Random Forest (Balanced)'] = rf

# 3. XGBoost com scale_pos_weight
print("Treinando 3/3: XGBoost...")
scale_pos_weight_calc = (len(y_train) - y_train.sum()) / y_train.sum()

xgb = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.08,
    scale_pos_weight=scale_pos_weight_calc,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
modelos['XGBoost (Weighted)'] = xgb

# Gerando predições e probabilidades no conjunto de teste
for nome, modelo in modelos.items():
    predicoes[nome] = modelo.predict(X_test)
    probabilidades[nome] = modelo.predict_proba(X_test)[:, 1]
    
print("Todos os modelos foram treinados com sucesso!")


## 5. Avaliação Comparativa de Desempenho

Métricas adotadas:
- **Recall (Sensibilidade)**: $\frac{TP}{TP + FN}$ — Percentual de fraudes reais identificadas.
- **Precisão**: $\frac{TP}{TP + FP}$ — De tudo o que o modelo acusou como fraude, quanto realmente era.
- **F1-Score**: Média harmônica entre Precisão e Recall.
- **AUPRC (Area Under Precision-Recall Curve)**: Métrica mais informativa que ROC-AUC para classes desbalanceadas.


In [ ]:
metricas_lista = []

for nome in modelos.keys():
    y_pred = predicoes[nome]
    y_prob = probabilidades[nome]
    
    rec = recall_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    # Matriz de confusão para extrair falsos negativos e falsos positivos
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    metricas_lista.append({
        'Modelo': nome,
        'Recall (Fraude)': rec,
        'Precisão (Fraude)': prec,
        'F1-Score': f1,
        'PR-AUC (Avg Precision)': pr_auc,
        'ROC-AUC': roc_auc,
        'Fraudes Detectadas (TP)': tp,
        'Fraudes Perdidas (FN)': fn,
        'Alarmes Falsos (FP)': fp
    })

df_comparativo = pd.DataFrame(metricas_lista).set_index('Modelo')
df_comparativo.sort_values(by='PR-AUC (Avg Precision)', ascending=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Curva ROC
for nome, y_prob in probabilidades.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f"{nome} (AUC = {auc:.4f})", lw=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.6)
axes[0].set_title("Curva ROC (Receiver Operating Characteristic)", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Taxa de Falsos Positivos (FPR)")
axes[0].set_ylabel("Taxa de Verdadeiros Positivos (Recall)")
axes[0].legend(loc="lower right")

# 2. Curva Precision-Recall (Padrão Ouro para dados desbalanceados)
for nome, y_prob in probabilidades.items():
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    axes[1].plot(recall, precision, label=f"{nome} (PR-AUC = {pr_auc:.4f})", lw=2)

axes[1].set_title("Curva Precisão x Recall", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Recall (Taxa de Fraudes Detectadas)")
axes[1].set_ylabel("Precisão (Confiabilidade dos Alarmes)")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()


## 6. Otimização do Limiar de Decisão (*Threshold Tuning*)

Por padrão, os classificadores utilizam o limiar de probabilidade $p = 0.5$. Em detecção de fraude, podemos ajustar esse limiar dependendo da estratégia do negócio:
- Se o custo de uma fraude não detectada for altíssimo, reduzimos o limiar para aumentar o **Recall**.
- Se o custo de investigar um alarme falso for elevado, ajustamos para o ponto de equilíbrio ótimo do **F1-Score**.


In [ ]:
# Ajuste de limiar para o XGBoost
y_prob_xgb = probabilidades['XGBoost (Weighted)']
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_xgb)

# Calculando F1 para cada limiar
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
melhor_idx = np.argmax(f1_scores)
melhor_threshold = thresholds[melhor_idx]
melhor_f1 = f1_scores[melhor_idx]

print(f"Limiar padrão: 0.5000")
print(f"Limiar ideal para maximizar F1: {melhor_threshold:.4f} (F1 = {melhor_f1:.4f})")

# Visualização da curva de ajuste
plt.figure(figsize=(10, 5))
plt.plot(thresholds, precisions[:-1], 'b--', label='Precisão', lw=2)
plt.plot(thresholds, recalls[:-1], 'g-', label='Recall', lw=2)
plt.plot(thresholds, f1_scores[:-1], 'r:', label='F1-Score', lw=2)
plt.axvline(melhor_threshold, color='black', linestyle='-.', alpha=0.7, label=f'Threshold Ótimo ({melhor_threshold:.2f})')
plt.title("Precisão, Recall e F1 em função do Limiar de Decisão (XGBoost)", fontsize=13, fontweight='bold')
plt.xlabel("Limiar de Decisão (Threshold)")
plt.ylabel("Pontuação")
plt.legend()
plt.grid(True)
plt.show()

# Testando predições com o novo limiar
y_pred_otimizado = (y_prob_xgb >= melhor_threshold).astype(int)
print("\nRelatório de Classificação com Limiar Otimizado:")
print(classification_report(y_test, y_pred_otimizado, target_names=['Normal', 'Fraude']))


## 7. Explicabilidade do Modelo (XAI) com SHAP

Modelos de caixa-preta (*black-box*) como XGBoost e Random Forest precisam ser explicados para que analistas de risco e órgãos reguladores compreendam o porquê de cada bloqueio.

O **SHAP (SHapley Additive exPlanations)** decompõe cada predição em contribuições aditivas de cada variável.


In [ ]:
# Usando TreeExplainer do SHAP com o modelo XGBoost
explainer = shap.TreeExplainer(xgb)

# Selecionamos uma amostra do conjunto de teste para cálculo rápido e limpo
X_test_sample = X_test.sample(n=1000, random_state=42)
shap_values = explainer(X_test_sample)

# 1. SHAP Beeswarm Plot (Visão Global)
plt.figure(figsize=(10, 8))
plt.title("Impacto Global das Variáveis no Risco de Fraude (SHAP Beeswarm)", fontsize=14, fontweight='bold', pad=15)
shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.tight_layout()
plt.show()


### Explicabilidade Local: Por que esta transação específica foi marcada como Fraude?

Vamos inspecionar uma transação que é comprovadamente fraude e observar como cada feature empurrou o score para cima (vermelho) ou para baixo (azul).


In [ ]:
# Identificando uma transação fraudulenta detectada pelo modelo
idx_fraude_real = y_test[y_test == 1].index[0]
amostra_fraude = X_test.loc[[idx_fraude_real]]

# Cálculo do SHAP para essa transação específica
shap_fraude_individual = explainer(amostra_fraude)

print(f"Analisando Transação ID: {idx_fraude_real}")
print(f"Probabilidade prevista pelo modelo: {xgb.predict_proba(amostra_fraude)[0, 1]:.2%}")

# Gráfico Waterfall
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_fraude_individual[0], max_display=10, show=False)
plt.title("Diagnóstico de Fraude Individual: Contribuição de cada Variável", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


## 8. Conclusões e Próximos Passos

### Principais Aprendizados:
1. **Acurácia Enganosa**: Em problemas com 0.17% de positivos, métricas como **PR-AUC**, **Recall** e **F1-Score** são as únicas que refletem o impacto real no negócio.
2. **Ponderação de Classes vs Reamostragem**: Ajustar o peso de penalização (`class_weight` / `scale_pos_weight`) preserva a integridade de dados originais e evita o custo computacional de inflar os dados.
3. **Threshold Tuning**: Reduzir ou refinar o limiar de decisão permite calibrar a tolerância a risco da instituição financeira.
4. **Governança e Transparência**: O SHAP permite que cada transação bloqueada venha acompanhada de seus motivos primários (ex.: `V14`, `V10` e `V12` desviantes), facilitando auditorias e suporte ao cliente.
